# HW01-C — Airflow Scheduled Pipeline

A pipeline that only runs when you remember to click it is a chore.

Here you turn the SQL work into an Airflow DAG. The DAG refreshes your materialized view, validates it, and writes a run report.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- Airflow DAGs: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html
- Airflow Variables: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/variables.html
- Airflow best practices: https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [2]:
from pathlib import Path
import os, re, textwrap

PROJECT = Path.cwd()
for path in ['dags', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(exist_ok=True)

student_id = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', student_id.lower())
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'
STUDENT_SCHEMA = f'student_{safe_student}'
DAG_ID, STUDENT_SCHEMA

('qbc12_hw01_arshia_ataei_airbnb_pipeline', 'student_arshia_ataei')

## 1. DAG design

Build this chain:

```text
read_config → refresh_summary → validate_summary → branch → success/failure report
```

Database credentials must come from Airflow Variables.

In [ ]:
# TODO 1.1
# Create dags/<DAG_ID>.py.
# Include imports, DAG metadata, make_engine(), and a read_config task.
# Use Airflow Variables for DB credentials.

from datetime import datetime, timedelta
from airflow import DAG
from airflow.decorators import task
from airflow.models import Variable
from sqlalchemy import create_engine, text

default_args = {
    'owner': 'arshia_ataei',
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

with DAG(
    dag_id='qbc12_hw01_arshia_ataei_airbnb_pipeline',
    default_args=default_args,
    schedule='@daily',
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=['qbc12', 'airbnb', 'hw01'],
):

    # Creating engine 
    def make_engine():
        user = Variable.get("QBC12_DB_USER")
        password = Variable.get("QBC12_DB_PASSWORD")
        host = Variable.get("QBC12_DB_HOST")
        port = Variable.get("QBC12_DB_PORT")
        db = Variable.get("QBC12_DB_NAME")

        engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}')
        return engine

    #used as other tasks settings
    @task
    def read_config():
        return{
            "student_schema" : "student_arshia_ataei",
            "mv_name" : "mv_airbnb_neighbourhood_summary",
        }

## 2. Refresh task

The refresh task should recreate your materialized view in Postgres. Do not move the full dataset into Python.

In [ ]:
# TODO 2.1
# Add refresh_summary(config).
# It should create/refresh your materialized view and indexes.
# Return a small dict, not a dataframe.

from datetime import datetime

@task
def refresh_summary(config):
    schema = config["student_schema"]
    mv_name = config["mv_name"] 

    engin = make_engine()

    # Creating Materialized View IF NOT created before (from HW01_B)
    with engin.begin() as conn:
        conn.execute(text(f'DROP MATERIALIZED VIEW IF EXISTS "{schema}"."{mv_name}"'))
        conn.execute(text(f'''
            CREATE MATERIALIZED VIEW "{schema}"."{mv_name}" AS
            WITH calendar_30 AS (
                SELECT listing_id, available, price
                FROM core.calendar_day
                WHERE date >= CURRENT_DATE - INTERVAL '30 days'
            ),
            review_counts AS (
                SELECT listing_id, COUNT(*) AS total_reviews
                FROM core.review
                GROUP BY listing_id
            )
            SELECT 
                core.listing.neighbourhood_id AS neighbourhood,
                COUNT(core.listing.listing_id) AS num_listings,
                AVG(core.listing.listing_price) AS avg_price,
                PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY core.listing.listing_price) AS median_price,
                AVG(core.listing.minimum_nights) AS avg_minimum_nights,
                COALESCE(SUM(review_counts.total_reviews), 0) AS total_reviews,
                CASE 
                    WHEN COUNT(core.listing.listing_id) > 0 
                    THEN COALESCE(SUM(review_counts.total_reviews), 0)::float / COUNT(core.listing.listing_id)
                    ELSE 0 
                END AS reviews_per_listing,
                COALESCE(AVG(CASE WHEN calendar_30.available = 't' THEN 1.0 ELSE 0.0 END), 0) AS availability_30_rate,
                COALESCE(AVG(CASE WHEN calendar_30.available = 't' THEN 1.0 ELSE 0.0 END), 0) * 365 AS availability_365_rate
            FROM core.listing
            LEFT JOIN calendar_30 ON core.listing.listing_id = calendar_30.listing_id
            LEFT JOIN review_counts ON core.listing.listing_id = review_counts.listing_id
            GROUP BY core.listing.neighbourhood_id
            ORDER BY num_listings DESC
        '''))

        conn.execute(text(f'CREATE INDEX IF NOT EXISTS idx_mv_neighbourhood ON "{schema}"."{mv_name}" (neighbourhood)'))
        conn.execute(text(f'CREATE INDEX IF NOT EXISTS idx_mv_num_listings ON "{schema}"."{mv_name}" (num_listings DESC)'))
    
    # counting rwos to return a summary
    with engin.begin() as conn:
        result = conn.execute(text(f'SELECT COUNT(*) FROM "{schema}"."{mv_name}"'))
        row_count = result.scalar()

    return{
        "materialized_view": f"{schema}.{mv_name}",
        "row_count" : row_count,
        "refreshed_at" : str(datetime.now())
    }


## 3. Validation task

Required checks:

- row_count > 0
- null_neighbourhoods == 0
- bad_prices == 0
- bad_availability == 0

In [ ]:
# TODO 3.1
# Add validate_summary(config).
# Return a dict containing each check and passed=True/False.

@task
def validate_summary(config):
    engine = make_engine()
    schema = config["student_schema"]
    mv_name = config["mv_name"]
    
    # calculating all validating variables
    with engine.connect() as conn:
        row_count = conn.execute(text(f'SELECT COUNT(*) FROM "{schema}"."{mv_name}"')).scalar()
        null_neighbourhoods = conn.execute(text(f'SELECT COUNT(*) FROM "{schema}"."{mv_name}" WHERE neighbourhood IS NULL')).scalar()
        bad_prices = conn.execute(text(f'SELECT COUNT(*) FROM "{schema}"."{mv_name}" WHERE avg_price < 0')).scalar()
        bad_availability = conn.execute(text(f'SELECT COUNT(*) FROM "{schema}"."{mv_name}" WHERE availability_30_rate < 0 OR availability_30_rate > 1')).scalar()

    validation = {}

    validation["row_counts_pos"] = row_count > 0
    validation["no_null_neighbourhood"] = null_neighbourhoods == 0
    validation["no_bad_prices"] = bad_prices == 0
    validation["no_bad_availibility"] = bad_availability == 0

    # ckecking all validations all togathar
    validation["passed"] = all([
    validation["row_counts_pos"],
    validation["no_null_neighbourhood"],
    validation["no_bad_prices"],
    validation["no_bad_availibility"],
    ])
    return validation


## 4. Branching and reports

Success and failure should not look the same.

Use `@task.branch` to choose the report path.

In [ ]:
# TODO 4.1
# Add choose_report_path, write_success_report, and write_failure_report.
# Failure report should raise ValueError after writing the report.

@task.branch
def choose_path(validation_result):
    if validation_result['passed']:
        return 'success_report'
    else:
        return 'failure_report'
    
@task
def success_report():
    print("Pipeline WORKS!")
    return {"status": "success"}

@task
def failure_report():
    print("Pipeline failed!")
    raise ValueError("Validation failed")

# at the end theres DAG chains al below:
config = read_config()
refreshed = refresh_summary(config)
validated = validate_summary(config)
branch = choose_path(validated)

refreshed >> validated >> branch
branch >> success_report()
branch >> failure_report()

In [5]:
# Syntax check. This is not a full Airflow run.
import py_compile

dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
assert dag_path.exists(), f'Missing DAG file: {dag_path}'
py_compile.compile(str(dag_path), doraise=True)
print('DAG compiles:', dag_path)

DAG compiles: c:\Arshia\AI\MLOps_Bootcamp\homework\hw01_airbnb\HW01_C\dags\qbc12_hw01_arshia_ataei_airbnb_pipeline.py


## 5. Shared Airflow run

In shared Airflow:

1. find your DAG
2. unpause it
3. trigger it manually
4. inspect Graph view
5. inspect logs
6. confirm the materialized view was refreshed

Screenshots:

```text
screenshots/airflow_dag_graph.png
screenshots/airflow_success_run.png
```

In [3]:
# TODO 5.1
# Write reports/hw01_c_airflow.md.
# Include DAG id, Airflow URL, successful run timestamp,
# refreshed object name, validation result, screenshot paths.

report = f"""# HW01-C Airflow Scheduled Pipeline Report

## DAG Information
- DAG ID: `qbc12_hw01_arshia_ataei_airbnb_pipeline`
- Schedule: Daily
- Schema: `student_arshia_ataei`
- Materialized View: `mv_airbnb_neighbourhood_summary`

## Airflow
- URL: http://185.50.38.163:33013
- DAG file: dags/qbc12_hw01_arshia_ataei_airbnb_pipeline.py

## Screenshots
- Graph: screenshots/airflow_dag_graph.png
- Success: screenshots/airflow_success_run.png
"""
Path('reports/hw01_c_airflow.md').write_text(report)

433

In [4]:
for file in [f'dags/{DAG_ID}.py', 'reports/hw01_c_airflow.md']:
    assert Path(file).exists(), f'Missing {file}'
print('Basic deliverables exist.')

Basic deliverables exist.


## Commit

```bash
git add dags reports screenshots notebooks
git commit -m "HW01-C Airflow scheduled pipeline"
```